In [0]:
#-------------------------------------------------------------------------------------------#
# Data criacao: 15/11/2025                                                                  #
# Objetivo:                                                                                 #
#   Alerta de ordens pendentes criação de KPI                                               #              
#   ordens pendentes, quantidade de itens e informações de clientes para envio de alerta por#
#   email e telefone de quem tem o cadastro                                                 #
#   Origem camada silver: customer                                                          #
#   Destino camada gold: orders_penging                                                     #
#-------------------------------------------------------------------------------------------#
# 1. Criar uma tabela temporária com os dados da tabela silver                              #
# 2. Criar uma tabela gold com os dados da tabela silver                                    #
# 4. Importar as funções desejadas                                                          #
#-------------------------------------------------------------------------------------------#

In [0]:
import sys
import importlib

# 1. Garante que alterações no arquivo .py sejam carregadas automaticamente
%load_ext autoreload
%autoreload 2

# 2. Adiciona a pasta pai ('Codigo') com o prefixo '/Workspace'
import os; caminho_codigo = os.path.abspath("../..")
if caminho_codigo not in sys.path:
    sys.path.append(caminho_codigo)

# 3. Importa e força o recarregamento do módulo
import funcoes_diversas.funcoes_util
importlib.reload(funcoes_diversas.funcoes_util)

# 4. Importa as funções desejadas
from funcoes_diversas.funcoes_util import buscar_caminho, gerar_tabela_temporaria



In [0]:
mapa_criado = gerar_tabela_temporaria(["customers","orders"],"silver")

View 'tmp_silver_customers' criada com sucesso!
View 'tmp_silver_orders' criada com sucesso!


In [0]:
%sql
describe tmp_silver_orders

col_name,data_type,comment
order_id,int,null
customer_id,int,null
order_status,int,null
status_descricao,string,null
order_date,date,null
required_date,date,null
shipped_date,string,null
staff_id,int,null
store_name,string,null
city,string,null


In [0]:
df_orders_pending_gold = spark.sql("""
    with pending as(
    select ORD.customer_id,
           ORD.order_date,
           ORD.store_name,
           ORD.status_descricao,
           sum(ORD.quantity) as Quantidade
    from tmp_silver_orders as ORD
    where 1 = 1
      and UPPER(ORD.status_descricao) = 'PENDING'
      group by ORD.customer_id,
           ORD.order_date,
           ORD.store_name,
           ORD.status_descricao)
    select ORD.customer_id,
           ORD.order_date,
           ORD.store_name,
           ORD.status_descricao,
           ORD.Quantidade,
           CUS.first_name,
            CUS.email,
           CUS.phone          
    FROM pending as ORD
    INNER JOIN tmp_silver_customers as CUS
           ON ORD.customer_id = CUS.customer_id
  """)


customer_id,order_date,store_name,status_descricao,Quantidade,first_name,email,phone
170,2018-04-17,Santa Cruz Bikes,Pending,4,Regine,regine.gonzales@gmail.com,(805) 763-4045
110,2018-04-13,Santa Cruz Bikes,Pending,3,Ollie,ollie.zimmerman@yahoo.com,(657) 648-2863
39,2018-04-17,Baldwin Bikes,Pending,8,Janetta,janetta.aguirre@aol.com,(717) 670-2634
80,2018-04-22,Baldwin Bikes,Pending,5,Sarai,sarai.mckee@msn.com,(716) 912-8110
56,2018-04-28,Rowlett Bikes,Pending,2,Lolita,lolita.mosley@hotmail.com,(281) 363-3309
5,2018-04-17,Santa Cruz Bikes,Pending,4,Charolette,charolette.rice@msn.com,(916) 381-6003
67,2018-04-17,Santa Cruz Bikes,Pending,1,Tommie,tommie.melton@gmail.com,(916) 802-2952
138,2018-04-27,Santa Cruz Bikes,Pending,4,Jone,jone.bernard@hotmail.com,(657) 536-5165
43,2018-04-29,Rowlett Bikes,Pending,8,Mozelle,mozelle.carter@aol.com,(281) 489-9656


In [0]:
gold_path = buscar_caminho("gold")
# transformando consulta em uma tabela Silver 
df_orders_pending_gold.write\
    .mode("overwrite")\
    .format("delta")\
    .option("overwriteSchema", "true")\
    .save(f"{gold_path}/orders_pending")